In [10]:
import pandas as pd

df = pd.read_csv("osc_sheet_11_with_industry_score.csv")
# or
# df = pd.read_csv("final_dataset_with_industry_score.csv")
target_col = "Closing Price.1"

X = df.drop(columns=[target_col])
y = df[target_col]
X = X.drop(columns=[
    "Company Name",
    "ISIN code",
    "NSE symbol",
    "Date","Industry group","Date.1"
], errors="ignore")
from sklearn.model_selection import train_test_split


print(df.shape)
df.head()




(998, 30)


,Company Name,Industry group,ISIN code,NSE symbol,Date,Total income from continuing operations,Net sales,Total expenses,Interest expenses,Depreciation,...,Closing Price.1,Shares Outstanding,Market Capitalisation,EPS,P/E,Number of Transactions,Enterprise value,Industry P/E,Beta,industry_score
0,3M India Ltd.,"Plastic furniture, floorings & miscellaneous i...",INE470A01017,3MINDIA,31-12-2022,943.41,932.22,818.55,0.56,13.88,...,23145.65,11265070,26073.736745,346.895314,66.722291,267,24891.636745,52.52,0.815759,-0.216968
1,A B B India Ltd.,"Generators, transformers & switchgears",INE117A01022,ABB,31-12-2022,2496.92,2426.91,2178.04,7.24,26.82,...,2826.55,211908375,59896.961736,29.890749,94.562703,1956,56780.971736,78.35,0.768015,-0.194812
2,A C C Ltd.,Cement,INE012A01025,ACC,31-12-2022,4577.66,4536.97,4348.65,18.83,171.56,...,1969.35,187787263,36981.884639,37.249065,52.869783,8437,30181.494639,48.83,0.976876,-0.201063
3,A D F Foods Ltd.,Processed foods,INE982B01027,ADFFOODS,31-12-2022,102.34,99.77,79.48,0.16,1.44,...,763.05,21972719,1676.628323,23.524626,32.436222,372,1585.448323,67.56,1.414542,-0.138728
4,A G I Greenpac Ltd.,Glass & glassware,INE415A01038,AGI,31-12-2022,570.40,567.30,529.48,13.13,30.25,...,326.05,64697381,2109.458108,24.643965,13.230420,1510,3226.098108,33.34,1.149549,-0.215826


In [11]:
import pandas as pd
import numpy as np

# =====================
# 1. Load data
# =====================
df = pd.read_csv("osc_sheet_11_with_industry_score.csv")

print("Original shape:", df.shape)

# =====================
# 2. Define target
# =====================
price_col = "Closing Price.1"

# Create binary target
# Investable (1): price >= median
# Non-investable (0): price < median
threshold = df[price_col].median()
df["investable"] = (df[price_col] >= threshold).astype(int)

# =====================
# 3. Feature / target split
# =====================
X = df.drop(columns=[
    price_col,
    "investable",
    "Company Name",
    "ISIN code",
    "NSE symbol",
    "Date",
    "Date.1",
    "Industry group"
], errors="ignore")

y = df["investable"]

print("Features shape:", X.shape)
print("Target distribution:")
print(y.value_counts())

# =====================
# 4. Handle missing values
# =====================
# Random Forest cannot handle NaNs
X = X.fillna(X.median(numeric_only=True))

# =====================
# 5. Train-test split
# =====================
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# =====================
# 6. Train Random Forest
# =====================
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

rf.fit(X_train, y_train)

# =====================
# 7. Evaluation
# =====================
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# =====================
# 8. Feature importance
# =====================
feature_importance = pd.Series(
    rf.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print("\nTop 15 Important Features:")
print(feature_importance.head(15))



Original shape: (998, 30)
Features shape: (998, 23)
Target distribution:
investable
1    499
0    499
Name: count, dtype: int64

Accuracy: 0.925
ROC-AUC: 0.9828

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.92      0.92       100
           1       0.92      0.93      0.93       100

    accuracy                           0.93       200
   macro avg       0.93      0.93      0.92       200
weighted avg       0.93      0.93      0.92       200


Confusion Matrix:
[[92  8]
 [ 7 93]]

Top 15 Important Features:
Closing Price                                   0.252306
EPS                                             0.136398
Shares Outstanding                              0.119178
Earnings per share before extraordinary item    0.100016
P/E                                             0.068318
Market Capitalisation                           0.056609
Enterprise value                                0.040576
Beta                   

In [12]:
import pandas as pd
COMPANY_COL="Company Name"
# --- Train table ---
train_df = pd.DataFrame({
    'Company': df.loc[X_train.index, COMPANY_COL],
    'Set': 'Train',
    'Investable': y_train.values
})

# --- Test table ---
test_df = pd.DataFrame({
    'Company': df.loc[X_test.index, COMPANY_COL],
    'Set': 'Test',
    'Investable': y_test.values
})

# Combine
full_df = pd.concat([train_df, test_df], ignore_index=True)

# Sort nicely
full_df = full_df.sort_values(by=['Set', 'Investable'], ascending=[True, False])

print(full_df)



                              Company    Set  Investable
798  Spandana Sphoorty Financial Ltd.   Test           1
802                D C M Shriram Ltd.   Test           1
803           Veedol Corporation Ltd.   Test           1
804                 H D F C Bank Ltd.   Test           1
805                 Shree Cement Ltd.   Test           1
..                                ...    ...         ...
788            Timex Group India Ltd.  Train           0
790      Dhani Services Ltd. [Merged]  Train           0
791               Greaves Cotton Ltd.  Train           0
793         Embassy Developments Ltd.  Train           0
796                 Valor Estate Ltd.  Train           0

[998 rows x 3 columns]


In [13]:
X_no_price = X.drop(columns=['Closing Price'])

Xtr, Xte, ytr, yte = train_test_split(
    X_no_price, y, test_size=0.2, random_state=42
)

rf.fit(Xtr, ytr)
print("Accuracy (no price):", accuracy_score(yte, rf.predict(Xte)))
print("ROC-AUC (no price):", roc_auc_score(yte, rf.predict_proba(Xte)[:,1]))


Accuracy (no price): 0.925
ROC-AUC (no price): 0.9813832449204284
